# Advanced analysis (exploratory only)

After EDA and data-quality checks, explore player/team segments with K-Means, project standardized features with PCA, inspect team-game feature importance where an outcome and eligible features exist, and compare seasons. Clusters and importances are descriptive, not validated predictions.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'backend' / 'app').is_dir())
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from backend.app import eda

DATA_DIRS = [ROOT / 'data' / 'processed', ROOT / 'data' / 'raw', ROOT / 'Raw']
players_result = eda.players_analysis(DATA_DIRS)
teams_result = eda.teams_analysis(DATA_DIRS)
print('Player summary available:', players_result['available'])
print('Team summary available:', teams_result['available'])

In [ ]:
correlation_result = eda.correlations_analysis(DATA_DIRS)
for table, values in correlation_result.get('correlations', {}).items():
    matrix = pd.DataFrame(values['matrix'], index=values['columns'], columns=values['columns'])
    px.imshow(matrix, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
              title=f'{table.title()} numeric feature correlations').show()
print('Outcome relationships:', correlation_result.get('relationships', {}))
outlier_rows = [{'table': table, 'feature': feature, **summary}
                for table, features in correlation_result.get('outliers', {}).items()
                for feature, summary in features.items()]
if outlier_rows:
    outliers = pd.DataFrame(outlier_rows).nlargest(20, 'outlier_count')
    px.bar(outliers, x='outlier_count', y='feature', color='table', orientation='h',
           title='Features with the most IQR outliers').show()
else:
    print('No numeric outlier summaries are available.')

In [ ]:
def cluster_summary(records, name, feature_names, clusters=3):
    frame = pd.DataFrame(records)
    if frame.empty or not set(feature_names).issubset(frame.columns) or len(frame) < clusters:
        print(f'{name}: insufficient available rows/features for clustering')
        return None
    features = frame[feature_names].apply(pd.to_numeric, errors='coerce')
    usable = features.notna().all(axis=1)
    frame, features = frame.loc[usable].copy(), features.loc[usable]
    if len(frame) < clusters:
        print(f'{name}: insufficient complete observations')
        return None
    scaled = StandardScaler().fit_transform(features)
    frame['cluster'] = KMeans(n_clusters=clusters, random_state=42, n_init=10).fit_predict(scaled)
    projection = PCA(n_components=2).fit_transform(scaled)
    frame['PC1'], frame['PC2'] = projection[:, 0], projection[:, 1]
    px.scatter(frame, x='PC1', y='PC2', color=frame['cluster'].astype(str),
               hover_name=name, title=f'{name.title()} segments (K-Means + PCA)').show()
    return frame

player_records = players_result.get('players', []) if players_result.get('available') else []
player_features = ['points', 'rebounds', 'assists', 'minutes']
player_segments = cluster_summary(
    [{'player': row['player'], **row['metrics']} for row in player_records],
    'player', player_features)
team_records = teams_result.get('teams', []) if teams_result.get('available') else []
team_features = ['average_points_scored', 'average_points_conceded', 'win_percentage']
team_segments = cluster_summary(team_records, 'team', team_features)

In [ ]:
sources = eda._source_files(DATA_DIRS)
team_path = eda._find_source(sources, ('teamstatisticsextended', 'teamstatistics'))
if team_path:
    team_games = eda._read(team_path)
    outcome_col = eda._find_column(team_games, ('win',))
    if outcome_col:
        target = pd.to_numeric(team_games[outcome_col], errors='coerce')
        if target.notna().sum() == 0:
            target = team_games[outcome_col].astype('string').str.lower().map(
                {'true': 1, 'yes': 1, 'w': 1, 'false': 0, 'no': 0, 'l': 0})
        numeric = team_games.select_dtypes(include='number')
        excluded = {'gameid', 'teamid', 'opponentteamid', 'win', 'winner', 'teamscore',
                    'opponentscore', 'plusminuspoints', 'seasonwins', 'seasonlosses'}
        columns = [column for column in numeric if eda._key(column) not in excluded
                   and not eda._key(column).endswith('score')]
        coverage = numeric[columns].notna().mean() if columns else pd.Series(dtype=float)
        columns = [column for column in columns if coverage[column] >= 0.75]
        features = numeric[columns].replace([np.inf, -np.inf], np.nan)
        features = features.fillna(features.median())
        usable = target.notna() & features.notna().all(axis=1)
        if usable.sum() >= 20 and target[usable].nunique() > 1 and columns:
            sample_index = features.loc[usable].index
            if len(sample_index) > 20000:
                sample_index = sample_index[np.linspace(0, len(sample_index) - 1, 20000, dtype=int)]
            model = RandomForestClassifier(n_estimators=200, random_state=42,
                                           class_weight='balanced', n_jobs=-1)
            model.fit(features.loc[sample_index], target.loc[sample_index].astype(int))
            importance = pd.DataFrame({'feature': columns,
                                       'importance': model.feature_importances_}).nlargest(20, 'importance')
            px.bar(importance, x='importance', y='feature', orientation='h',
                   title='Exploratory team-game outcome feature importance').show()
        else:
            print('Feature importance unavailable: too few complete rows or only one outcome class.')
    else:
        print('Feature importance unavailable: no team-game win/outcome column.')
else:
    print('Feature importance unavailable: team statistics dataset not found.')

In [ ]:
seasons = eda.seasons_analysis(DATA_DIRS)
season_scores = pd.DataFrame(seasons.get('game_scoring_by_season', []))
if not season_scores.empty:
    value = 'average_points_per_team' if 'average_points_per_team' in season_scores else 'points'
    px.line(season_scores, x='season', y=value, markers=True,
            title='Season comparison: scoring trend').show()
team_seasons = pd.DataFrame(seasons.get('team_performance_by_season', []))
if not team_seasons.empty:
    px.box(team_seasons, x='season', y='win_percentage', points=False,
           title='Distribution of team win rates by season').show()
print('Unavailable analyses:', seasons.get('unavailable_analyses', []))